In [1]:
# Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# I. Dataset Source : UniRef50

In [2]:
df = pd.read_csv("/u/mdmc/enyanduk/internship_areasciencepark/Dataframes/DPCFam/uniref50_pfam_valid.csv")
df.head()

,uniref50_id,pfam_ids,pfam_ranges
0,G5B0U1,PF07679,15159-15237
1,G5B0U1,PF07679,11109-11197
2,G5B0U1,PF07679,10241-10328
3,G5B0U1,PF07679,2348-2427
4,G5B0U1,PF07679,31048-31127


In [3]:
# Info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14554301 entries, 0 to 14554300
Data columns (total 3 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   uniref50_id  object
 1   pfam_ids     object
 2   pfam_ranges  object
dtypes: object(3)
memory usage: 333.1+ MB


In [4]:
# How long are the pfam_ids?
pfam_id_lengths = df["pfam_ids"].str.len()
print("Unique lengths of pfam_ids:", pfam_id_lengths.unique())

Unique lengths of pfam_ids: [7]


**Great, all pfam ids comply with PFAM naming convention, no issue here.**

In [5]:
# Unique pfam_ids
unique_pfam_ids = df["pfam_ids"].unique()
print("Number of unique pfam_ids:", len(unique_pfam_ids))

Number of unique pfam_ids: 18189


In [6]:
# Check whether there is any pfam_ids starting with CL
pfam_ids_starting_with_CL = df[df["pfam_ids"].str.startswith("CL")]
print("Number of pfam_ids starting with CL:", len(pfam_ids_starting_with_CL))

Number of pfam_ids starting with CL: 0


**Okay, we do not have PFam clans here**.

In [7]:
# We create a dataframe of those ids:
unique_labels = sorted(unique_pfam_ids)
df1 = pd.DataFrame({"pfam_id": unique_pfam_ids})
df1.head()

,pfam_id
0,PF07679
1,PF02818
2,PF18362
3,PF00069
4,PF00041


In [8]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18189 entries, 0 to 18188
Data columns (total 1 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   pfam_id  18189 non-null  object
dtypes: object(1)
memory usage: 142.2+ KB


# II. Collecting Pfam labels from DPCFam

In [9]:
df_dpcfam = pd.read_csv("/u/mdmc/enyanduk/internship_areasciencepark/Dataframes/DPCFam/Merged/dpcfam_all_mcs_props.csv")
df_dpcfam.head()

,mcid,size_uniref50,avg_len,std_avg_len,lc_percent,cc_percent,dis_percent,tm,pfam_da,da_percent,avg_ov_percent,overlap_label
0,MC1,17931,185.68,28.77,4.72,0.00,18.44,0.01,PF13614,44.23,80.82,equivalent
1,MC3,28,215.14,25.44,5.08,0.38,30.15,0.00,PF14631,8571.43,24.57,shifted
2,MC4,617,59.91,6.07,4.99,0.00,1.87,1.26,PF03600,62.84,7.54,shifted
3,MC5,26,253.19,23.11,4.88,0.00,7.51,0.00,PF12937,6800.00,16.89,extended
4,MC7,32,65.53,3.30,3.39,0.00,0.10,0.00,PF01243,7407.41,6.27,shifted


In [10]:
# How long are the pfam_ids?
dpcfam_pfam_id_lengths = df_dpcfam["pfam_da"].str.len()
print("Unique lengths of pfam_ids:", dpcfam_pfam_id_lengths.unique())

Unique lengths of pfam_ids: [  7  15  23  31  39  79  63  47 183  55  71 111]


**For composite Pfam dominant architectures, we used "-" symbole to split them**.

In [11]:
# We care about the dominant architecture : 
# Split all pfam_da entries on "-" to get individual labels, then flatten
all_labels = df_dpcfam["pfam_da"].str.split("-").explode().str.strip()
print("Total individual labels in pfam_da:", len(all_labels))
# Unique labels
unique_labels = all_labels.unique()
print("Unique labels in pfam_da:", len(unique_labels))
# How many UNKNOWN entries are there?
unknown_count = (all_labels == "UNKNOWN").sum()
print(f"Number of UNKNOWN entries: {unknown_count}")
# Remove UNKNOWN entries for counting PF/CL
labels_no_unknown = all_labels[all_labels != "UNKNOWN"]

pf_count = labels_no_unknown.str.startswith("PF").sum()
cl_count = labels_no_unknown.str.startswith("CL").sum()
other_count = len(labels_no_unknown) - pf_count - cl_count

print(f"Total individual labels (excl. UNKNOWN): {len(labels_no_unknown)}")
print(f"PF (Pfam families): {pf_count}")
print(f"CL (Pfam clans):    {cl_count}")
print(f"Other:               {other_count}")

Total individual labels in pfam_da: 85265
Unique labels in pfam_da: 12804
Number of UNKNOWN entries: 33179
Total individual labels (excl. UNKNOWN): 52086
PF (Pfam families): 52086
CL (Pfam clans):    0
Other:               0


In [12]:
# Great, in DPCFam, we have only families, no clans.
# We create a dataframe of those unique ids(containing UNKNOWN as well):
dpcfam_pfam_labels = sorted(unique_labels)
df2 = pd.DataFrame({"pfam_id": dpcfam_pfam_labels})
df2.head()

,pfam_id
0,PF00001
1,PF00002
2,PF00003
3,PF00004
4,PF00005


In [13]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12804 entries, 0 to 12803
Data columns (total 1 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   pfam_id  12804 non-null  object
dtypes: object(1)
memory usage: 100.2+ KB


In [14]:
# How long are the pfam_ids?
final_dpcfam_pfam_id_lengths = df2["pfam_id"].str.len()
print("Unique lengths of pfam_ids:", final_dpcfam_pfam_id_lengths.unique())

Unique lengths of pfam_ids: [7]


**Great, we fixed the problem. All Pfam IDs have the same lengths**.

# III. Collecting PFam labels from DPCStruct

In [15]:
df_dpcstruct = pd.read_csv("/u/mdmc/enyanduk/internship_areasciencepark/Dataframes/DPCStruct/dpcstruct_mcs_properties.csv")
df_dpcstruct.head()

,mc_id,mc_size,len_aa,len_std,len_ratio,plddt,disorder,tmscore,lddt,pident,pfam_score,pfam_da
0,MC0,43,112.02,12.36,0.11,78.53,0.29,0.54,0.65,23.74,0.0,UNKNOWN
1,MC1,19,68.89,7.43,0.11,90.81,0.28,0.77,0.82,25.87,0.0,UNKNOWN
2,MC2,19,368.05,52.78,0.14,85.00,0.30,0.46,0.56,19.60,0.0,UNKNOWN
3,MC3,18,346.39,34.27,0.10,82.09,0.25,0.55,0.49,14.00,0.0,UNKNOWN
4,MC4,12,135.17,13.06,0.10,86.24,0.22,0.69,0.62,23.25,0.0,UNKNOWN


In [16]:
# Info
df_dpcstruct.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28246 entries, 0 to 28245
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   mc_id       28246 non-null  object 
 1   mc_size     28246 non-null  int64  
 2   len_aa      28246 non-null  float64
 3   len_std     28246 non-null  float64
 4   len_ratio   28246 non-null  float64
 5   plddt       28246 non-null  float64
 6   disorder    28246 non-null  float64
 7   tmscore     28246 non-null  float64
 8   lddt        28246 non-null  float64
 9   pident      28246 non-null  float64
 10  pfam_score  28246 non-null  float64
 11  pfam_da     28246 non-null  object 
dtypes: float64(9), int64(1), object(2)
memory usage: 2.6+ MB


In [17]:
# How long are the pfam_ids?
dpcstruct_pfam_id_lengths = df_dpcstruct["pfam_da"].str.len()
print("Unique lengths of pfam_ids:", dpcstruct_pfam_id_lengths.unique())

Unique lengths of pfam_ids: [ 7  6 13 14 15 38 37 29 22 21 23 20 34 36 35 27 30 31 28 39]


In [18]:
# We care about the dominant architecture : 
# Split all pfam_da entries on "-" to get individual labels, then flatten
all_dpcstruct_labels = df_dpcstruct["pfam_da"].str.split("-").explode().str.strip()
print("Total individual labels in pfam_da:", len(all_dpcstruct_labels))
# Unique labels
unique_dpcstruct_labels = all_dpcstruct_labels.unique()
print("Unique labels in pfam_da:", len(unique_dpcstruct_labels))
# How many UNKNOWN entries are there?
unknown_dpcstruct_count = (all_dpcstruct_labels == "UNKNOWN").sum()
print(f"Number of UNKNOWN entries: {unknown_dpcstruct_count}")
# Remove UNKNOWN entries for counting PF/CL
dpcstruct_labels_no_unknown = all_dpcstruct_labels[all_dpcstruct_labels != "UNKNOWN"]

pf_count = dpcstruct_labels_no_unknown.str.startswith("PF").sum()
cl_count = dpcstruct_labels_no_unknown.str.startswith("CL").sum()
other_count = len(dpcstruct_labels_no_unknown) - pf_count - cl_count

print(f"Total individual labels (excl. UNKNOWN): {len(dpcstruct_labels_no_unknown)}")
print(f"PF (Pfam families): {pf_count}")
print(f"CL (Pfam clans):    {cl_count}")
print(f"Other:               {other_count}")

Total individual labels in pfam_da: 36107
Unique labels in pfam_da: 5560
Number of UNKNOWN entries: 13823
Total individual labels (excl. UNKNOWN): 22284
PF (Pfam families): 9943
CL (Pfam clans):    12341
Other:               0


In [19]:
# In DPCStruct, we have both families and clans.
# We create a dataframe of those unique ids(containing UNKNOWN as well):
dpcstruct_pfam_labels = sorted(unique_dpcstruct_labels)
df3 = pd.DataFrame({"pfam_id": dpcstruct_pfam_labels})
df3.head()

,pfam_id
0,CL0001
1,CL0003
2,CL0004
3,CL0005
4,CL0007


In [20]:
#Info
df3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5560 entries, 0 to 5559
Data columns (total 1 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   pfam_id  5560 non-null   object
dtypes: object(1)
memory usage: 43.6+ KB


In [21]:
# How long are the pfam_ids?
final_dpcstruct_pfam_id_lengths = df3["pfam_id"].str.len()
print("Unique lengths of pfam_ids:", final_dpcstruct_pfam_id_lengths.unique())

Unique lengths of pfam_ids: [6 7]


**Okay, we know that Clans are named CL+4digits : 6 letters and for families,it's PF+5digits : 7 letters(same as UNKNOWN)**

# IV. Putting everything together

In [22]:
# Is there any intersection between dpcfam and dpcstruct pfam_ids?
dpcstruct_pfam_ids = set(df3["pfam_id"])
dpcfam_pfam_ids = set(df2["pfam_id"])
intersection_pfam_ids = dpcstruct_pfam_ids.intersection(dpcfam_pfam_ids)
print("Number of intersecting pfam_ids between dpcfam and dpcstruct:", len(intersection_pfam_ids))
# Print some of the intersecting pfam_ids
print("Some intersecting pfam_ids:", list(intersection_pfam_ids)[:10])

Number of intersecting pfam_ids between dpcfam and dpcstruct: 3972
Some intersecting pfam_ids: ['PF07559', 'PF07947', 'PF07009', 'PF05800', 'PF18058', 'PF17253', 'PF05929', 'PF16881', 'PF03344', 'PF05502']


In [23]:
# Great, now we can concatenate the three dataframes
df = pd.concat([df1, df2, df3], ignore_index=True).drop_duplicates().reset_index(drop=True)
df.head()

,pfam_id
0,PF07679
1,PF02818
2,PF18362
3,PF00069
4,PF00041


In [24]:
# We assign pfam_type based on the prefix of pfam_id
df["pfam_type"] = df["pfam_id"].apply(
    lambda x: "Family" if x.startswith("PF") else ("Clan" if x.startswith("CL") else "Other")
)

print(f"df shape: {df.shape}")
print(f"Unique PF families: {(df['pfam_type'] == 'Family').sum()}")
print(f"Unique CL clans:    {(df['pfam_type'] == 'Clan').sum()}")
print(f"Unknown:       {(df['pfam_type'] == 'Other').sum()}")
df.head(10)

df shape: (19352, 2)
Unique PF families: 18734
Unique CL clans:    617
Unknown:       1


,pfam_id,pfam_type
0,PF07679,Family
1,PF02818,Family
2,PF18362,Family
3,PF00069,Family
4,PF00041,Family
5,PF09042,Family
6,PF13927,Family
7,PF00047,Family
8,PF13018,Family
9,PF12951,Family


In [25]:
# Info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19352 entries, 0 to 19351
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   pfam_id    19352 non-null  object
 1   pfam_type  19352 non-null  object
dtypes: object(2)
memory usage: 302.5+ KB


In [26]:
# We save the dataframe as a csv file
df.to_csv("/u/mdmc/enyanduk/internship_areasciencepark/Dataframes/DPC/dpc_pfam_ids.csv", index=False)